# 02 — Feature Engineering
**Football Betting Integrity Monitor**

Reads from `s_vladislavkivaev.mart_matches_clean` and builds the heavier, state/ordering-dependent
features in Python: per-league-season z-scores, cross-market divergence, and per-league rolling
temporal features — the work that needs ordering/state and doesn't belong in SQL.

**Design rule carried throughout this notebook**
Features are z-scored **per league-season**. The COVID season (19/20) is therefore already its own
baseline group — it's z-scored against itself like every other season, so no masking is needed here.
COVID rows are kept in the data; the `is_covid_season` flag is carried forward as a lever for
`03_modeling` (drop from Isolation Forest training, or use as a feature).

**Handoff to `03_modeling` (the H2-vs-H3 spine)**
This notebook deliberately preserves **both** representations of each signal:
- **natural-unit** values (scaled globally in modeling) -> feed the **universal** Isolation Forest ->
  mid-tier's wide-but-normal spreads sit far from the global mean -> over-flagged. *This is H2.*
- **per-league-season z-scored** (`*_z`) values -> feed the **tier-aware** leg -> each league centred on
  its own normal -> balanced false-positive rate. *This is H3.*

If we only kept the z-scored version, H2's failure mode would vanish — so Block 2 keeps both.

## Block 0 — Connect & load

SQLAlchemy engine from `.env` (`sslmode=require`), read the mart, then sanity-check shape, coverage,
and that `is_covid_season` survived the dbt pass-through.

In [1]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

pd.set_option("display.max_columns", 120)

In [2]:
load_dotenv()  # reads .env in the project root

DB_HOST     = os.environ["DB_HOST"]
DB_PORT     = os.environ.get("DB_PORT", "5432")
DB_NAME     = os.environ["DB_NAME"]
DB_USER     = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_SCHEMA   = os.environ.get("DB_SCHEMA", "s_vladislavkivaev")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"sslmode": "require"},
)

In [3]:
df = pd.read_sql(f"SELECT * FROM {DB_SCHEMA}.mart_matches_clean", engine)
print(df.shape)
df.head()

(8915, 65)


,match_date,home_team,away_team,home_goals,away_goals,full_time_result,b365_open_h,b365_open_d,b365_open_a,ps_open_h,ps_open_d,ps_open_a,max_open_h,max_open_d,max_open_a,avg_open_h,avg_open_d,avg_open_a,b365_close_h,b365_close_d,b365_close_a,ps_close_h,ps_close_d,ps_close_a,max_close_h,max_close_d,max_close_a,avg_close_h,avg_close_d,avg_close_a,league,season,tier,is_covid_season,b365_drift_h,b365_drift_d,b365_drift_a,pinnacle_drift_h,pinnacle_drift_d,pinnacle_drift_a,opening_spread_h,opening_spread_d,opening_spread_a,closing_spread_h,closing_spread_d,closing_spread_a,max_opening_spread,max_closing_spread,spread_change_h,spread_change_d,spread_change_a,max_spread_change,b365_vs_market_h,b365_vs_market_d,b365_vs_market_a,implied_prob_sum_open,implied_prob_sum_close,season_quintile,b365_vs_ps_close_h,b365_vs_ps_close_d,b365_vs_ps_close_a,margin_tightened,overround_change,home_shortened,home_win
0,2019-08-16,Bayern Munich,Hertha,2,2,D,1.14,8.00,15.00,1.19,7.73,15.30,1.25,8.30,17.50,1.18,7.55,15.04,1.14,8.0,15.0,1.18,8.49,17.38,1.25,9.10,19.25,1.17,8.00,15.67,D1,1920,elite,True,0.00,0.00,0.00,-0.01,0.76,2.08,0.07,0.75,2.46,0.08,1.10,3.58,2.46,3.58,0.01,0.35,1.12,1.12,0.11,1.10,4.25,1.068860,1.068860,Q1\n(0-20%),-0.04,-0.49,-2.38,False,0.000000,False,False
1,2019-08-17,Dortmund,Augsburg,5,1,H,1.20,7.00,13.00,1.23,6.76,13.52,1.25,7.15,17.00,1.22,6.62,13.38,1.12,8.0,26.0,1.16,8.37,20.11,1.20,8.80,26.00,1.16,8.10,18.29,D1,1920,elite,True,-0.08,1.00,13.00,-0.07,1.61,6.59,0.03,0.53,3.62,0.04,0.70,7.71,3.62,7.71,0.01,0.17,4.09,4.09,0.08,0.80,0.00,1.053114,1.056319,Q1\n(0-20%),-0.04,-0.37,5.89,False,0.003205,True,True
2,2019-08-17,Freiburg,Mainz,3,0,H,2.25,3.25,3.40,2.23,3.45,3.44,2.26,3.49,3.65,2.20,3.37,3.36,2.55,3.3,2.7,2.74,3.30,2.77,2.74,3.38,2.96,2.61,3.29,2.78,D1,1920,elite,True,0.30,0.05,-0.70,0.51,-0.15,-0.67,0.06,0.12,0.29,0.13,0.09,0.18,0.29,0.18,0.07,-0.03,-0.11,0.07,0.19,0.08,0.26,1.046254,1.065558,Q1\n(0-20%),-0.19,0.00,-0.07,False,0.019303,False,True
3,2019-08-17,Leverkusen,Paderborn,3,2,H,1.25,6.00,12.00,1.29,6.07,10.57,1.31,6.40,12.25,1.28,5.97,10.27,1.22,6.5,11.0,1.27,6.69,10.45,1.29,7.05,13.00,1.25,6.52,10.70,D1,1920,elite,True,-0.03,0.50,-1.00,-0.02,0.62,-0.12,0.03,0.43,1.98,0.04,0.53,2.30,1.98,2.30,0.01,0.10,0.32,0.32,0.07,0.55,2.00,1.050000,1.064427,Q1\n(0-20%),-0.05,-0.19,0.55,False,0.014427,True,True
4,2019-08-17,Werder Bremen,Fortuna Dusseldorf,1,3,A,1.75,3.75,4.75,1.75,4.05,4.68,1.80,4.05,4.95,1.74,3.93,4.57,1.66,4.2,4.5,1.70,4.15,4.90,1.76,4.32,5.10,1.69,4.07,4.79,D1,1920,elite,True,-0.09,0.45,-0.25,-0.05,0.10,0.22,0.06,0.12,0.38,0.07,0.25,0.31,0.38,0.31,0.01,0.13,-0.07,0.13,0.10,0.12,0.60,1.048622,1.062727,Q1\n(0-20%),-0.04,0.05,-0.40,False,0.014106,True,False


In [4]:
# --- Sanity checks ---
df["match_date"] = pd.to_datetime(df["match_date"])
df = df.sort_values("match_date").reset_index(drop=True)

print("Rows:", len(df), "(source mart had 8,915)")
print("\nLeagues:", sorted(df["league"].unique()))
print("Seasons:", sorted(df["season"].unique()))

print("\nMatches per league x season:")
print(df.groupby(["league", "season"]).size().unstack(fill_value=0))

assert "is_covid_season" in df.columns, "is_covid_season missing — check the dbt pass-through"
print("\nCOVID rows:", int(df["is_covid_season"].sum()))

print("\nTier map:")
print(df[["tier", "league"]].drop_duplicates().sort_values(["tier", "league"]).to_string(index=False))

Rows: 8915 (source mart had 8,915)

Leagues: ['D1', 'E0', 'G1', 'T1']
Seasons: [np.int64(1920), np.int64(2021), np.int64(2122), np.int64(2223), np.int64(2324), np.int64(2425), np.int64(2526)]

Matches per league x season:
season  1920  2021  2122  2223  2324  2425  2526
league                                          
D1       306   306   306   306   306   306   306
E0       380   380   380   380   380   380   380
G1       240   239   240   238   240   233   236
T1       306   420   380   313   380   342   306

COVID rows: 1232

Tier map:
    tier league
   elite     D1
   elite     E0
mid_tier     G1
mid_tier     T1


In [5]:
# COVID flag. Under per-league-SEASON grouping, the 19/20 season is already its own
# baseline group, so it's isolated by the grouping itself — no masking needed in 02.
# Kept here as a lever for 03_modeling (drop from IF training, or use as a feature).
is_covid = df["is_covid_season"].astype(bool)
print(f"COVID rows: {int(is_covid.sum())} / {len(df)}")

COVID rows: 1232 / 8915


## Block 1 — The baseline engine (load-bearing)

One reusable function computes per-group mean/std and applies the z-score to every row in the group.
By default it uses all rows in each group (Option 1: each season, COVID included, is scored against
itself). It also accepts an optional `baseline_mask` to restrict which rows define the baseline —
unused in `02`, but available for `03_modeling`. Two guards:

- `std == 0` or `NaN` (thin season groups, esp. G1 / T1) -> z set to `0.0`
- group with no baseline stats -> z set to `0.0`

Default group key is `["league", "season"]`. A `["tier", "season"]` scheme is available for a
secondary view. Every downstream block calls this — so the normalization logic is debugged exactly once.

In [6]:
def zscore_by_group(df, cols, group_keys=("league", "season"),
                    baseline_mask=None, suffix="_z"):
    """
    Per-group z-score, optionally with a separate baseline population.

    Parameters
    ----------
    df : DataFrame
    cols : list[str]            columns to z-score
    group_keys : list[str]      grouping for the baseline (default league + season)
    baseline_mask : Series|None boolean aligned to df rows; True = row used to compute
                                mean/std. Default None = use all rows in each group.
    suffix : str                appended to each new z-score column

    Returns
    -------
    DataFrame : copy of df with one `{col}{suffix}` column per input col.
                Guards: std == 0 / NaN / missing-group  ->  z = 0.0
    """
    group_keys = list(group_keys)
    cols = list(cols)
    out = df.reset_index(drop=True).copy()

    # 1) compute stats from the chosen baseline population
    base = out if baseline_mask is None else out.loc[baseline_mask.values]
    agg = base.groupby(group_keys)[cols].agg(["mean", "std"])
    agg.columns = [f"{c}__{stat}" for c, stat in agg.columns]
    agg = agg.reset_index()

    # 2) attach group stats to every row (left join preserves row order)
    merged = out.merge(agg, on=group_keys, how="left", sort=False)

    # 3) z-score with guards
    for col in cols:
        mean = merged[f"{col}__mean"]
        std  = merged[f"{col}__std"]
        z = (merged[col] - mean) / std
        z = z.where(std.notna() & (std != 0), 0.0).fillna(0.0)  # degenerate/undefined -> 0
        out[f"{col}{suffix}"] = z.to_numpy()

    return out

In [7]:
# --- Smoke test on one column (Option 1: no baseline_mask) ---
_test = zscore_by_group(df, cols=["b365_drift_h"])

print(_test[["league", "season", "b365_drift_h", "b365_drift_h_z"]].head())

print("\nAny inf in z? ", bool(np.isinf(_test["b365_drift_h_z"]).any()))
print("Any NaN in z? ", bool(_test["b365_drift_h_z"].isna().any()))

# Centring check: mean of z across every league-season group should be ~0
chk = _test.groupby(["league", "season"])["b365_drift_h_z"].mean()
print("\nMax |mean of z| across groups (expect ~0):", round(chk.abs().max(), 6))

  league  season  b365_drift_h  b365_drift_h_z
0     E0    1920          0.00       -0.006810
1     E0    1920          0.00       -0.006810
2     E0    1920          0.08        0.132133
3     E0    1920          0.40        0.687904
4     E0    1920          0.20        0.340547

Any inf in z?  False
Any NaN in z?  False

Max |mean of z| across groups (expect ~0): 0.0


## Block 2 — Z-score the five signal families

Run each canonical anomaly signal through `zscore_by_group` so every model feature traces back to a
nameable concept (this is what keeps the SHAP story in `03` interpretable). The natural-unit columns
are **kept alongside** the new `*_z` columns — modeling needs both representations.

| Family | Source columns (natural-unit) | Anomaly meaning |
|---|---|---|
| **Drift** | `b365_drift_*`, `pinnacle_drift_*` | large unidirectional open->close move = informed money |
| **Spread** | `opening/closing_spread_*`, `spread_change_*`, `max_*_spread`, `max_spread_change` | books disagree = asymmetric information |
| **Implied-prob imbalance** | `implied_prob_sum_open/close`, `overround_change` | unusual overround = market inefficiency |
| **CLV / cross-book** | `b365_vs_market_*`, `b365_vs_ps_close_*` | one book off the consensus = different information |
| **Reversal (derived here)** | from `b365_drift_*` vs `pinnacle_drift_*` | sharp vs public book move opposite ways = steam-then-fade |

**Reversal is a proxy, stated honestly:** with only open+close snapshots we cannot observe a true
intra-window reversal. The defensible substitute is *direction disagreement* between the sharp book
(Pinnacle) and the public book (Bet365) on the same outcome. These flags are **booleans — not z-scored.**

In [8]:
# --- Define the five signal families (natural-unit movement columns from dbt) ---
drift_cols = ["b365_drift_h", "b365_drift_d", "b365_drift_a",
              "pinnacle_drift_h", "pinnacle_drift_d", "pinnacle_drift_a"]

spread_cols = ["opening_spread_h", "opening_spread_d", "opening_spread_a",
               "closing_spread_h", "closing_spread_d", "closing_spread_a",
               "spread_change_h", "spread_change_d", "spread_change_a",
               "max_opening_spread", "max_closing_spread", "max_spread_change"]

implied_cols = ["implied_prob_sum_open", "implied_prob_sum_close", "overround_change"]

clv_cols = ["b365_vs_market_h", "b365_vs_market_d", "b365_vs_market_a",
            "b365_vs_ps_close_h", "b365_vs_ps_close_d", "b365_vs_ps_close_a"]

families = {"drift": drift_cols, "spread": spread_cols,
            "implied": implied_cols, "clv": clv_cols}

# Defensive: only z-score columns that actually exist in the mart
present, missing = {}, {}
for fam, cols in families.items():
    present[fam] = [c for c in cols if c in df.columns]
    missing[fam] = [c for c in cols if c not in df.columns]
    flag = ("  MISSING: " + ", ".join(missing[fam])) if missing[fam] else ""
    print(f"{fam:8s} present: {len(present[fam])}/{len(cols)}{flag}")

drift    present: 6/6
spread   present: 12/12
implied  present: 3/3
clv      present: 6/6


In [9]:
# --- Reversal proxy (derived here; not in dbt) ---
# Sharp book (Pinnacle) and public book (Bet365) moved in OPPOSITE directions on the
# same outcome = steam-then-fade fingerprint. Boolean per outcome + an "any outcome" rollup.
# np.sign(0) == 0, so "no movement" never counts as a disagreement.
reversal_flags = []
for o in ["h", "d", "a"]:
    col = f"dir_disagree_{o}"
    df[col] = ((np.sign(df[f"b365_drift_{o}"]) * np.sign(df[f"pinnacle_drift_{o}"])) < 0).astype(int)
    reversal_flags.append(col)

df["dir_disagree_any"] = df[reversal_flags].max(axis=1)
reversal_cols = reversal_flags + ["dir_disagree_any"]

print("Reversal-flag rates (share of matches):")
print(df[reversal_cols].mean().round(3).to_string())

Reversal-flag rates (share of matches):
dir_disagree_h      0.061
dir_disagree_d      0.113
dir_disagree_a      0.080
dir_disagree_any    0.213


In [10]:
# --- Z-score the numeric families in one pass (natural-unit columns are kept) ---
to_z = present["drift"] + present["spread"] + present["implied"] + present["clv"]
df = zscore_by_group(df, cols=to_z)        # Option 1: per league-season, no baseline_mask

z_cols = [f"{c}_z" for c in to_z]
print(f"Created {len(z_cols)} z-scored columns from {len(to_z)} natural-unit columns.")
df[z_cols].describe().T[["mean", "std", "min", "max"]].round(3)

Created 27 z-scored columns from 27 natural-unit columns.


,mean,std,min,max
b365_drift_h_z,-0.0,0.997,-11.054,13.887
b365_drift_d_z,0.0,0.997,-11.110,15.095
b365_drift_a_z,0.0,0.997,-11.222,14.534
pinnacle_drift_h_z,0.0,0.957,-13.075,12.340
pinnacle_drift_d_z,0.0,0.957,-11.151,17.912
pinnacle_drift_a_z,0.0,0.957,-9.480,14.743
opening_spread_h_z,-0.0,0.998,-0.952,17.168
opening_spread_d_z,-0.0,0.998,-1.654,14.452
opening_spread_a_z,-0.0,0.998,-0.907,13.116
closing_spread_h_z,0.0,0.998,-0.929,14.879


In [11]:
# --- QA: both representations coexist; z-scores well-behaved ---
example = to_z[0]
print(f"Both representations present? {example} & {example}_z ->",
      (example in df.columns) and (f"{example}_z" in df.columns))

print("Any inf across z cols?", bool(np.isinf(df[z_cols].to_numpy()).any()))
print("Any NaN across z cols?", bool(df[z_cols].isna().to_numpy().any()))

# Pooled across all rows: within-group centring makes mean ~0, std ~1
summary = df[z_cols].agg(["mean", "std"]).T
print("\nMax |pooled mean| across z cols:", round(summary["mean"].abs().max(), 4))
print("Pooled std range:", round(summary["std"].min(), 3), "to", round(summary["std"].max(), 3))

Both representations present? b365_drift_h & b365_drift_h_z -> True
Any inf across z cols? False
Any NaN across z cols? False

Max |pooled mean| across z cols: 0.0
Pooled std range: 0.957 to 0.998


---
**Next — Block 3:** cross-market divergence. Collapse the per-outcome book disagreement into a single
*dispersion* magnitude (z-scored through the same engine) plus a *consensus-steam* boolean — the novel
feature flagged as the most interesting result in the project brief.

## Block 3 — Cross-market divergence (the novel feature)

Two genuinely new signals, kept separate because they mean different things:

- **Consensus steam** — when the sharp book (Pinnacle) and public book (Bet365) move the *same*
  direction on an outcome, how big is the concerted move? Both books steaming one way is the
  strongest informed-money fingerprint available from open+close data. (This is the complement of
  Block 2's `dir_disagree`: disagreement = books split; steam = books aligned and moving.)
- **Cross-market divergence magnitude** — collapse the per-outcome Bet365-vs-Pinnacle closing gap
  (`b365_vs_ps_close_*`) into a single per-match magnitude + the outcome it lands on. One book sitting
  far off the sharp consensus on one specific outcome is the candidate signal.

**Honesty note on overlap:** dbt already encodes a lot of dispersion (`closing_spread_*`,
`max_closing_spread`, `spread_change_*`). Block 3 doesn't re-derive those — it adds the two things
*not* yet present (concerted-move magnitude, collapsed divergence) and then **checks correlation
against the existing spread features**, so we keep these only if they carry independent signal.

In [12]:
# --- Consensus steam: concerted same-direction move across the two books ---
# When Bet365 and Pinnacle agree on direction, take their average drift; else 0.
steam_signed_cols = []
for o in ["h", "d", "a"]:
    b = df[f"b365_drift_{o}"]
    p = df[f"pinnacle_drift_{o}"]
    same_dir = (np.sign(b) * np.sign(p)) > 0          # both moved, same way
    df[f"steam_{o}"] = np.where(same_dir, (b + p) / 2.0, 0.0)
    steam_signed_cols.append(f"steam_{o}")

# Collapse to match level: the outcome with the largest concerted move
steam_abs = df[steam_signed_cols].abs()
pos = steam_abs.to_numpy().argmax(axis=1)
df["steam_abs"]    = steam_abs.to_numpy()[np.arange(len(df)), pos]      # magnitude
df["steam_signed"] = df[steam_signed_cols].to_numpy()[np.arange(len(df)), pos]  # signed
df["steam_outcome"] = pd.Series(["h", "d", "a"])[pos].to_numpy()
df.loc[df["steam_abs"] == 0, "steam_outcome"] = None    # no concerted move -> no outcome

print("Matches with a concerted move (steam_abs > 0):",
      round((df["steam_abs"] > 0).mean(), 3))
print("\nSteam outcome distribution (where present):")
print(df["steam_outcome"].value_counts(dropna=True).to_string())

Matches with a concerted move (steam_abs > 0): 0.822

Steam outcome distribution (where present):
steam_outcome
a    3863
h    2205
d    1256


In [13]:
# --- Cross-market divergence: collapse b365-vs-pinnacle closing gap to one magnitude ---
ps_cols = ["b365_vs_ps_close_h", "b365_vs_ps_close_d", "b365_vs_ps_close_a"]
ps_abs = df[ps_cols].abs()
pos = ps_abs.to_numpy().argmax(axis=1)

df["xmkt_div_abs"]    = ps_abs.to_numpy()[np.arange(len(df)), pos]        # magnitude
df["xmkt_div_signed"] = df[ps_cols].to_numpy()[np.arange(len(df)), pos]   # signed
df["xmkt_div_outcome"] = pd.Series(["h", "d", "a"])[pos].to_numpy()
df.loc[df["xmkt_div_abs"] == 0, "xmkt_div_outcome"] = None

print("Cross-market divergence outcome distribution:")
print(df["xmkt_div_outcome"].value_counts(dropna=True).to_string())

# z-score both new magnitudes through the same engine (per league-season)
df = zscore_by_group(df, cols=["steam_abs", "xmkt_div_abs"])
print("\nNew z-cols created: steam_abs_z, xmkt_div_abs_z")
print(df[["steam_abs_z", "xmkt_div_abs_z"]].describe().T[["mean", "std", "min", "max"]].round(3))

Cross-market divergence outcome distribution:
xmkt_div_outcome
a    3937
h    3021
d    1957

New z-cols created: steam_abs_z, xmkt_div_abs_z
                mean    std    min     max
steam_abs_z      0.0  0.998 -1.002  15.620
xmkt_div_abs_z   0.0  0.960 -1.028  16.006


In [14]:
# --- Do the new features carry signal independent of existing spread features? ---
check_cols = ["steam_abs_z", "xmkt_div_abs_z",
              "max_closing_spread_z", "max_spread_change_z",
              "b365_vs_ps_close_h_z", "b365_vs_ps_close_d_z", "b365_vs_ps_close_a_z"]
print("Correlation of new vs existing features:\n")
print(df[check_cols].corr().round(2))

Correlation of new vs existing features:

                      steam_abs_z  xmkt_div_abs_z  max_closing_spread_z  \
steam_abs_z                  1.00            0.40                  0.50   
xmkt_div_abs_z               0.40            1.00                  0.69   
max_closing_spread_z         0.50            0.69                  1.00   
max_spread_change_z          0.42            0.46                  0.79   
b365_vs_ps_close_h_z        -0.00           -0.11                 -0.01   
b365_vs_ps_close_d_z        -0.28           -0.49                 -0.43   
b365_vs_ps_close_a_z        -0.24           -0.78                 -0.48   

                      max_spread_change_z  b365_vs_ps_close_h_z  \
steam_abs_z                          0.42                 -0.00   
xmkt_div_abs_z                       0.46                 -0.11   
max_closing_spread_z                 0.79                 -0.01   
max_spread_change_z                  1.00                 -0.01   
b365_vs_ps_close_h_z  

In [15]:
# Decision: keep steam_abs_z (novel, independent); drop xmkt_div_abs_z from model set (redundant with b365_vs_ps_close_*)
# Keep xmkt_div_signed / xmkt_div_outcome as descriptive-only columns.
drop_from_model = ["xmkt_div_abs_z"]
print("Dropping from model feature set:", drop_from_model)
print("steam_abs_z max corr with existing spread features:",
      round(df[["steam_abs_z","max_closing_spread_z","max_spread_change_z"]].corr()["steam_abs_z"].drop("steam_abs_z").abs().max(), 2))

Dropping from model feature set: ['xmkt_div_abs_z']
steam_abs_z max corr with existing spread features: 0.5


## Block 4 — Per-league rolling temporal features

Turns the EDA draw-spike findings (Greece 22/23, Turkey 25/26) into features that fire *in real time*
rather than hardcoded facts. Two trailing signals per match, computed within `[league, season]`:

- **`roll_draw_rate`** — draw rate over the trailing 38 fixtures. A run of matches during a draw spike
  carries an elevated trailing rate → the spike becomes a per-match feature.
- **`roll_drift_mag`** — mean Bet365 drift magnitude over the trailing 38 fixtures. Captures whether
  the market has been unusually volatile in this league lately.

**Leakage guard:** every window is built with `.shift(1)` so the current match is **excluded from its
own window** — non-negotiable, since these could feed the supervised Random Forest leg in `03`.

**Window / NaN policy:** window = 38, `min_periods = 10` (estimate kicks in after 10 prior fixtures).
Early-season rows below that fall back to a leakage-safe trailing **expanding** mean; the single
first fixture of each season (no prior at all) gets the column's global mean as a neutral prior.
Both rolling features are then z-scored per league-season through the same engine — so "unusually
high recent draw rate *for this league-season*" becomes the anomaly signal.

In [16]:
# --- Sort for rolling (within-group order = chronological) and build per-match scalars ---
df = df.sort_values(["league", "season", "match_date"]).reset_index(drop=True)

df["is_draw"]   = (df["full_time_result"] == "D").astype(int)
df["drift_mag"] = df[["b365_drift_h", "b365_drift_d", "b365_drift_a"]].abs().mean(axis=1)

# Sanity: draw rate should be ~0.22–0.28; if 0.0, full_time_result isn't coded 'D'
print("Overall draw rate:", round(df["is_draw"].mean(), 3))
print("Draw rate by league:")
print(df.groupby("league")["is_draw"].mean().round(3).to_string())

Overall draw rate: 0.252
Draw rate by league:
league
D1    0.247
E0    0.236
G1    0.272
T1    0.259


In [17]:
WINDOW, MIN_PERIODS = 38, 10
g = df.groupby(["league", "season"])

# trailing window, current match excluded via shift(1)
df["roll_draw_rate"] = g["is_draw"].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=MIN_PERIODS).mean())
df["roll_drift_mag"] = g["drift_mag"].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=MIN_PERIODS).mean())

print("NaN after rolling:", df[["roll_draw_rate", "roll_drift_mag"]].isna().sum().to_dict())

# leakage-safe fallback for early-season rows: trailing expanding mean
df["roll_draw_rate"] = df["roll_draw_rate"].fillna(
    g["is_draw"].transform(lambda s: s.shift(1).expanding().mean()))
df["roll_drift_mag"] = df["roll_drift_mag"].fillna(
    g["drift_mag"].transform(lambda s: s.shift(1).expanding().mean()))

# first fixture of each season has no prior at all -> global-mean neutral prior
for c in ["roll_draw_rate", "roll_drift_mag"]:
    df[c] = df[c].fillna(df[c].mean())

print("NaN after fallback:", df[["roll_draw_rate", "roll_drift_mag"]].isna().sum().to_dict())

NaN after rolling: {'roll_draw_rate': 280, 'roll_drift_mag': 280}
NaN after fallback: {'roll_draw_rate': 0, 'roll_drift_mag': 0}


In [18]:
# z-score per league-season through the same engine
df = zscore_by_group(df, cols=["roll_draw_rate", "roll_drift_mag"])

print("Rolling z-cols:")
print(df[["roll_draw_rate_z", "roll_drift_mag_z"]].describe().T[["mean","std","min","max"]].round(3))

# Validation: peak trailing draw-rate per league-season — Greece 22/23 & Turkey 25/26 should rank high
print("\nTop trailing draw-rate periods (league-season, peak roll_draw_rate):")
print(df.groupby(["league","season"])["roll_draw_rate"].max()
        .sort_values(ascending=False).head(8).round(3).to_string())

Rolling z-cols:
                  mean    std    min     max
roll_draw_rate_z   0.0  0.998 -5.172  11.818
roll_drift_mag_z  -0.0  0.998 -4.983  10.122

Top trailing draw-rate periods (league-season, peak roll_draw_rate):
league  season
D1      1920      1.000
G1      2122      1.000
        2324      1.000
        2223      1.000
        1920      1.000
D1      2122      1.000
T1      2324      0.667
E0      1920      0.500


### Validation

In [19]:
# Validation, fixed: only count windows that are actually full (>= 38 prior fixtures)
full = df["roll_draw_rate"].notna()  # all rows now, so gate on a real window count instead
df["_win_n"] = (df.groupby(["league","season"])["is_draw"]
                  .transform(lambda s: s.shift(1).rolling(38, min_periods=38).count()))
stable = df[df["_win_n"] >= 38]

print("Peak trailing draw-rate over FULL 38-match windows (league-season):")
print(stable.groupby(["league","season"])["roll_draw_rate"].max()
            .sort_values(ascending=False).head(8).round(3).to_string())

# And the z-score view — which periods read as anomalous for their own league-season
print("\nPeak roll_draw_rate_z over full windows:")
print(stable.groupby(["league","season"])["roll_draw_rate_z"].max()
            .sort_values(ascending=False).head(8).round(3).to_string())
df = df.drop(columns="_win_n")

Peak trailing draw-rate over FULL 38-match windows (league-season):
league  season
E0      2526      0.500
D1      2021      0.474
T1      2526      0.447
        2021      0.447
        1920      0.447
D1      2425      0.447
G1      2526      0.447
E0      2021      0.447

Peak roll_draw_rate_z over full windows:
league  season
D1      2526      3.006
T1      2122      2.736
E0      2324      2.735
        2526      2.734
D1      2021      2.602
E0      2021      2.552
T1      2223      2.381
        1920      2.353


In [ ]:
# Peak inspection / denominator check
peak = df[(df["league"]=="D1") & (df["season"]==2526)].nlargest(1, "roll_draw_rate_z")
print(peak[["match_date","roll_draw_rate","roll_draw_rate_z"]].to_string(index=False))
print("Matches in D1 2526 so far:", ((df["league"]=="D1") & (df["season"]==2526)).sum())

match_date  roll_draw_rate  roll_draw_rate_z
2026-01-24        0.421053          3.005685
Matches in D1 2526 so far: 306


## Block 5 — Assemble the feature matrix + role tagging + leakage/QA

Every column is assigned exactly one **role**, recorded in a `feature_roles` dict (persisted in Block 6).
The two model sets are then *derived from roles*, never hardcoded:

- **Universal set (H2)** = `model_natural` + `model_both` → fed to the pooled Isolation Forest on
  globally-scaled natural-unit features → mid-tier's wide-but-normal spreads read as far-from-global → over-flags.
- **Tier-aware set (H3)** = `model_z` + `model_both` → per-league-season z-scored → each league centred → balanced FPR.

Roles:
- `id` — carried for traceback, never modelled (teams, date, scores, league/season/tier, result)
- `model_natural` / `model_z` — the two representations of the same signals
- `model_both` — scale-free booleans used in *both* model sets
- `label` — `home_win`, kept separate; predicts **result, not integrity** (optional supervised RF leg only)
- `descriptive` — carried for the investigation slide, not fed to any model
- `excluded` — dropped with cause on record (`xmkt_div_abs*`: redundant with `b365_vs_ps_close_*`)

**Leakage guards audited here:** no `label` or `id` column leaks into a model set; no NaN/inf in the
model sets; row count preserved.

### Defining roles

In [21]:
# --- Reconstruct the numeric family list from Block 2 (present = columns that existed in the mart) ---
family_natural = present["drift"] + present["spread"] + present["implied"] + present["clv"]   # 27

# Two representations of the same signals
model_natural = family_natural + ["steam_abs", "roll_draw_rate", "roll_drift_mag"]
model_z       = [f"{c}_z" for c in family_natural] + ["steam_abs_z", "roll_draw_rate_z", "roll_drift_mag_z"]

# Scale-free booleans -> both model sets. Granular per-outcome flags carry "which outcome";
# dir_disagree_any is their max (redundant) -> demoted to descriptive. Flip if you'd rather model the rollup.
model_both = ["dir_disagree_h", "dir_disagree_d", "dir_disagree_a"]

label_cols    = ["home_win"]
excluded_cols = ["xmkt_div_abs", "xmkt_div_abs_z"]   # redundant with b365_vs_ps_close_* (on record, Block 3)

id_cols = ["match_date", "home_team", "away_team", "home_goals", "away_goals",
           "full_time_result", "league", "season", "tier", "is_covid_season"]

# Build the role map, then sweep anything unassigned into 'descriptive'
feature_roles = {}
for role, cols in [("id", id_cols), ("model_natural", model_natural), ("model_z", model_z),
                   ("model_both", model_both), ("label", label_cols), ("excluded", excluded_cols)]:
    for c in cols:
        feature_roles[c] = role

unassigned = [c for c in df.columns if c not in feature_roles]
for c in unassigned:
    feature_roles[c] = "descriptive"

print("Unassigned -> descriptive:", unassigned)
print("\nColumns per role:")
print(pd.Series(feature_roles).value_counts().to_string())

Unassigned -> descriptive: ['b365_open_h', 'b365_open_d', 'b365_open_a', 'ps_open_h', 'ps_open_d', 'ps_open_a', 'max_open_h', 'max_open_d', 'max_open_a', 'avg_open_h', 'avg_open_d', 'avg_open_a', 'b365_close_h', 'b365_close_d', 'b365_close_a', 'ps_close_h', 'ps_close_d', 'ps_close_a', 'max_close_h', 'max_close_d', 'max_close_a', 'avg_close_h', 'avg_close_d', 'avg_close_a', 'season_quintile', 'margin_tightened', 'home_shortened', 'dir_disagree_any', 'steam_h', 'steam_d', 'steam_a', 'steam_signed', 'steam_outcome', 'xmkt_div_signed', 'xmkt_div_outcome', 'is_draw', 'drift_mag']

Columns per role:
descriptive      37
model_natural    30
model_z          30
id               10
model_both        3
excluded          2
label             1


### Leakage + QA Audit

In [22]:
universal_set  = model_natural + model_both
tier_aware_set = model_z + model_both

# 1) no label / id leaked into a model set
leak = (set(universal_set) | set(tier_aware_set)) & (set(label_cols) | set(id_cols))
assert not leak, f"LEAKAGE: id/label in a model set -> {leak}"

# 2) every model column actually exists
missing = [c for c in universal_set + tier_aware_set if c not in df.columns]
assert not missing, f"Missing model columns: {missing}"

# 3) z-set is NaN/inf free (guard guarantees this); natural-set may carry NaN from missing odds
z_bad = df[tier_aware_set].isna().to_numpy().any() or np.isinf(df[model_z].to_numpy()).any()
print("Tier-aware (z) set has NaN/inf?", bool(z_bad), "(expected: False)")

nat_nan = df[model_natural].isna().sum()
nat_nan = nat_nan[nat_nan > 0]
print("\nNatural-unit columns with NaN (handle by imputation in 03 if any):")
print("  none" if nat_nan.empty else nat_nan.to_string())
print("Natural-unit inf?", bool(np.isinf(df[model_natural].to_numpy()).any()))

# 4) row count preserved
print("\nRows:", len(df), "(source mart: 8,915)")

Tier-aware (z) set has NaN/inf? False (expected: False)

Natural-unit columns with NaN (handle by imputation in 03 if any):
b365_drift_h              27
b365_drift_d              27
b365_drift_a              27
pinnacle_drift_h         728
pinnacle_drift_d         728
pinnacle_drift_a         728
opening_spread_h           1
opening_spread_d           1
opening_spread_a           1
spread_change_h            1
spread_change_d            1
spread_change_a            1
max_opening_spread         1
max_spread_change          1
implied_prob_sum_open     27
overround_change          27
b365_vs_ps_close_h       680
b365_vs_ps_close_d       680
b365_vs_ps_close_a       680
Natural-unit inf? False

Rows: 8915 (source mart: 8,915)


### Final Summary

In [23]:
print(f"Universal set (H2):  {len(universal_set)} features  ({len(model_natural)} natural + {len(model_both)} boolean)")
print(f"Tier-aware set (H3): {len(tier_aware_set)} features  ({len(model_z)} z + {len(model_both)} boolean)")
print(f"Label (RF only):     {label_cols}")
print(f"Excluded:            {excluded_cols}")
print(f"\nTotal columns in df: {df.shape[1]}  |  Identifiers: {sum(v=='id' for v in feature_roles.values())}"
      f"  |  Descriptive: {sum(v=='descriptive' for v in feature_roles.values())}")

Universal set (H2):  33 features  (30 natural + 3 boolean)
Tier-aware set (H3): 33 features  (30 z + 3 boolean)
Label (RF only):     ['home_win']
Excluded:            ['xmkt_div_abs', 'xmkt_div_abs_z']

Total columns in df: 113  |  Identifiers: 10  |  Descriptive: 37


### Finalize roles

Add the `pinnacle_missing` coverage flag (built before imputation so it records the true gap),
promote `margin_tightened` to a modelled boolean, and rebuild the two model sets from roles.

In [24]:
# Coverage flag — built BEFORE imputation so it records the real gap, not a filled value
df["pinnacle_missing"] = df["pinnacle_drift_h"].isna().astype(int)
print("pinnacle_missing rate:", round(df["pinnacle_missing"].mean(), 3))
print(df.groupby(["league", "season"])["pinnacle_missing"].mean()
        .loc[lambda s: s > 0].round(2).to_string())

# Booleans used in BOTH model sets (scale-free): reversal flags + sharp-money + coverage
model_both = ["dir_disagree_h", "dir_disagree_d", "dir_disagree_a",
              "margin_tightened", "pinnacle_missing"]
for c in model_both:
    df[c] = df[c].fillna(0).astype(int)   # margin_tightened can be NaN where opening odds missing

# update role map + rebuild sets from roles
feature_roles["margin_tightened"] = "model_both"
feature_roles["pinnacle_missing"] = "model_both"
universal_set  = model_natural + model_both
tier_aware_set = model_z + model_both

leak = (set(universal_set) | set(tier_aware_set)) & (set(label_cols) | set(id_cols))
assert not leak, f"LEAKAGE: {leak}"
print(f"\nUniversal (H2): {len(universal_set)} | Tier-aware (H3): {len(tier_aware_set)} | model_both: {len(model_both)}")

pinnacle_missing rate: 0.082
league  season
D1      2526      0.51
E0      2526      0.45
G1      1920      0.08
        2021      0.07
        2526      0.77
T1      1920      0.01
        2122      0.00
        2324      0.00
        2425      0.01
        2526      0.57

Universal (H2): 35 | Tier-aware (H3): 35 | model_both: 5


## Block 6 — Feature dictionary + persist

Build a per-feature dictionary (family, representation, tail shape, which model set, imputation plan)
and persist three artifacts to `data/processed/`: the feature matrix (`features.parquet`), the role map
(`feature_roles.json`), and the dictionary (`feature_dictionary.csv`). `03_modeling` and the SHAP
labelling read from these instead of hardcoding column lists.

In [25]:
model_cols = list(dict.fromkeys(universal_set + tier_aware_set))  # ordered union

def classify(col):
    is_z = col.endswith("_z")
    base = col[:-2] if is_z else col
    rep = "boolean" if col in model_both else ("z" if is_z else "natural")
    if "drift" in base:                                         fam, tail = "drift", "two_tailed"
    elif "spread_change" in base or base == "max_spread_change":fam, tail = "spread", "two_tailed"
    elif "spread" in base:                                      fam, tail = "spread", "upper"
    elif base.startswith("implied_prob") or base == "overround_change":
                                                                fam, tail = "implied_prob_imbalance", "two_tailed"
    elif base.startswith("b365_vs_market"):                     fam, tail = "clv_crossbook", "upper"
    elif base.startswith("b365_vs_ps"):                         fam, tail = "clv_crossbook", "two_tailed"
    elif base.startswith("dir_disagree"):                       fam, tail = "reversal", "n/a"
    elif base.startswith("steam"):                              fam, tail = "steam", "upper"
    elif base.startswith("roll"):                               fam, tail = "rolling_temporal", "two_tailed"
    elif base == "margin_tightened":                            fam, tail = "margin", "n/a"
    elif base == "pinnacle_missing":                            fam, tail = "coverage", "n/a"
    else:                                                       fam, tail = "other", "n/a"
    return base, fam, rep, tail

def impute_plan(base, rep):
    if rep != "natural":
        return "none (boolean; z-guard zeros NaN)"
    if base.startswith("pinnacle") or base.startswith("b365_vs_ps"):
        return "per-league-season median + pinnacle_missing flag (structural 25/26 gap)"
    return "per-league-season median"

rows = []
for col in model_cols:
    base, fam, rep, tail = classify(col)
    rows.append({"column": col, "family": fam, "representation": rep, "tail_shape": tail,
                 "in_universal_H2": col in universal_set, "in_tier_aware_H3": col in tier_aware_set,
                 "imputation": impute_plan(base, rep)})

feat_dict = (pd.DataFrame(rows)
             .sort_values(["family", "representation", "column"]).reset_index(drop=True))
print(feat_dict.to_string(index=False))

                  column                 family representation tail_shape  in_universal_H2  in_tier_aware_H3                                                              imputation
        b365_vs_market_a          clv_crossbook        natural      upper             True             False                                                per-league-season median
        b365_vs_market_d          clv_crossbook        natural      upper             True             False                                                per-league-season median
        b365_vs_market_h          clv_crossbook        natural      upper             True             False                                                per-league-season median
      b365_vs_ps_close_a          clv_crossbook        natural two_tailed             True             False per-league-season median + pinnacle_missing flag (structural 25/26 gap)
      b365_vs_ps_close_d          clv_crossbook        natural two_tailed             True     

### Persist

In [26]:
import json, os

OUT = "../data/processed"
os.makedirs(OUT, exist_ok=True)

df.to_parquet(f"{OUT}/features.parquet", index=False)   # needs pyarrow: pip install pyarrow

with open(f"{OUT}/feature_roles.json", "w") as f:
    json.dump({
        "roles": feature_roles,
        "universal_set_H2": universal_set,
        "tier_aware_set_H3": tier_aware_set,
        "model_both": model_both,
        "label": label_cols,
        "excluded": excluded_cols,
        "notes": {
            "covid": "z-scored per league-season; 19/20 is its own baseline group, no masking",
            "pinnacle_gap": "structural ~50% missing in 25/26 (football-data upload lag, cuts off ~Jan 2026); flagged via pinnacle_missing",
            "imputation_runs_in": "03_modeling",
        },
    }, f, indent=2)

feat_dict.to_csv(f"{OUT}/feature_dictionary.csv", index=False)

print("Saved:")
print(f"  {OUT}/features.parquet           {df.shape}")
print(f"  {OUT}/feature_roles.json")
print(f"  {OUT}/feature_dictionary.csv     {len(feat_dict)} model features")

Saved:
  ../data/processed/features.parquet           (8915, 114)
  ../data/processed/feature_roles.json
  ../data/processed/feature_dictionary.csv     65 model features
